# Config-Driven Web Scraping Pipeline

Scrape financial PDFs from Moroccan institutional websites using the **config-driven** approach:
- Reads site configurations from `config/scrapers/sites.json`
- Crawls listing pages to discover PDF links
- Downloads PDFs to `data/collected_data/documents/{site}/`
- Dispatches to ingestion pipeline (optional)

**Configured sources:**
- Bank Al-Maghrib (BKAM)
- Cour des Comptes (CDC)
- Haut-Commissariat au Plan (HCP)
- AMMC (Autorité du Marché des Capitaux)
- Ministère de l'Économie et des Finances (MEF)

---

In [1]:
import sys
import os
import json
from pathlib import Path
from _setup import PROJECT_ROOT

import pandas as pd

print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\lenovo\Desktop\Pfe 2026\financial-advisor


## 1. Load Scraper Configuration

In [2]:
CONFIG_FILE = PROJECT_ROOT / "config" / "scrapers" / "sites.json"

with open(CONFIG_FILE, encoding="utf-8") as f:
    sites = json.load(f)

print(f"Total sites configured: {len(sites)}\n")
print(f"{'Name':<20} {'Display Name':<50} {'Enabled':<8}")
print("-" * 80)
for site in sites:
    status = "YES" if site.get("enabled", True) else "NO"
    print(f"{site['name']:<20} {site['display_name']:<50} {status}")

Total sites configured: 5

Name                 Display Name                                       Enabled 
--------------------------------------------------------------------------------
bank_al_maghrib      Bank Al-Maghrib (BKAM)                             YES
cdc                  Cour des Comptes (CDC)                             NO
hcp                  Haut-Commissariat au Plan (HCP)                    NO
ammc                 Autorité Marocaine du Marché des Capitaux (AMMC)   YES
finances_gov         Ministère de l'Économie et des Finances (MEF)      YES


## 2. Discover PDFs from a Site

Crawl listing pages and extract PDF links (without downloading).

In [3]:
import requests
import re
import time
import random
from bs4 import BeautifulSoup
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

def discover_pdfs(listing_url: str, base_url: str, pdf_pattern: str = ".pdf") -> list:
    """Crawl a listing page and return all discovered PDF links."""
    entries = []
    seen = set()
    
    try:
        resp = requests.get(listing_url, headers=HEADERS, verify=False, timeout=20)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        
        for link in soup.find_all("a", href=True):
            href = link["href"]
            full_url = href if href.startswith("http") else base_url + href
            
            if href.lower().endswith(pdf_pattern) or pdf_pattern in href:
                if full_url not in seen:
                    seen.add(full_url)
                    title = link.get_text(strip=True) or Path(href).stem
                    entries.append({"title": title[:100], "url": full_url})
    except Exception as e:
        print(f"  Error: {e}")
    
    return entries

In [4]:
# Discover PDFs from enabled sites
enabled_sites = [s for s in sites if s.get("enabled", True)]

for site in enabled_sites:
    print(f"\n[{site['display_name']}]")
    print("=" * 60)
    
    total_pdfs = 0
    for url in site["listing_urls"]:
        pdfs = discover_pdfs(url, site["base_url"], site.get("pdf_link_pattern", ".pdf"))
        total_pdfs += len(pdfs)
        print(f"  {url}")
        print(f"  → Found {len(pdfs)} PDFs")
        for pdf in pdfs[:3]:  # Show first 3
            print(f"    • {pdf['title'][:60]}")
        if len(pdfs) > 3:
            print(f"    ... and {len(pdfs) - 3} more")
    
    print(f"  Total: {total_pdfs} PDFs discovered")
    time.sleep(1)  # Polite delay between sites


[Bank Al-Maghrib (BKAM)]
  https://www.bkam.ma/Publications-et-recherche/Publications-institutionnelles/Rapport-annuel-presente-a-sm-le-roi
  → Found 10 PDFs
    • (PDF)
    • (PDF)
    • (PDF)
    ... and 7 more
  https://www.bkam.ma/Publications-et-recherche/Documents-d-analyse/Rapport-sur-la-politique-monetaire
  → Found 21 PDFs
    • (PDF)
    • (PDF)
    • (PDF)
    ... and 18 more
  https://www.bkam.ma/Publications-et-recherche/Publications-institutionnelles/Rapport-annuel-sur-la-stabilite-financiere
  → Found 10 PDFs
    • (PDF)
    • (PDF)
    • (PDF)
    ... and 7 more
  Total: 41 PDFs discovered

[Autorité Marocaine du Marché des Capitaux (AMMC)]
  https://www.ammc.ma/fr/publications
  → Found 10 PDFs
    • RAPPORT PROFIL DES INVESTISSEURS VFR T1 26-1_0.pdf
    • LE MARCHE DES CAPITAUX EN CHIFFRES 2025 VFR.pdf
    • GUIDE SUR LA GESTION SOUS MANDAT D’INSTRUMENTS FINANCIERS_VF
    ... and 7 more
  Error: 404 Client Error: Not Found for url: https://www.ammc.ma/fr/rapports-ann

## 3. Download a PDF

Download a single discovered PDF to the local data folder.

In [5]:
import hashlib

DOCS_DIR = PROJECT_ROOT / "data" / "collected_data" / "documents"

def download_pdf(pdf_url: str, title: str, site_name: str, output_folder: str) -> str:
    """Download a PDF and return its local path."""
    dest_dir = DOCS_DIR / output_folder
    dest_dir.mkdir(parents=True, exist_ok=True)
    
    slug = re.sub(r"[^\w\s-]", "", title[:50]).strip().replace(" ", "_")
    url_hash = hashlib.md5(pdf_url.encode()).hexdigest()[:8]
    filename = f"{site_name}_{slug}_{url_hash}.pdf"
    dest = dest_dir / filename
    
    if dest.exists():
        print(f"  Already exists: {filename}")
        return str(dest)
    
    resp = requests.get(pdf_url, headers=HEADERS, stream=True, timeout=60, verify=False)
    resp.raise_for_status()
    
    with open(dest, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
    
    size_mb = dest.stat().st_size / (1024 * 1024)
    print(f"  [OK] Downloaded: {filename} ({size_mb:.1f} MB)")
    return str(dest)

# Example: download first PDF from AMMC
# Uncomment to run:
# ammc = next(s for s in sites if s["name"] == "ammc")
# pdfs = discover_pdfs(ammc["listing_urls"][0], ammc["base_url"])
# if pdfs:
#     path = download_pdf(pdfs[0]["url"], pdfs[0]["title"], "ammc", "ammc")
#     print(f"  Saved to: {path}")

## 4. Full Pipeline: Scrape → Download → Ingest

Runs the complete scraping pipeline for a single site.

In [6]:
from ingestion import IngestionPipeline

def scrape_and_ingest_site(site_config: dict, max_pdfs: int = 3):
    """
    Full pipeline for one site:
    1. Discover PDFs from listing pages
    2. Download each PDF
    3. Ingest into ChromaDB
    """
    name = site_config["name"]
    base_url = site_config["base_url"]
    output_folder = site_config.get("output_folder", name)
    pdf_pattern = site_config.get("pdf_link_pattern", ".pdf")
    
    print(f"\n{'='*60}")
    print(f"  Scraping: {site_config['display_name']}")
    print(f"{'='*60}")
    
    # 1. Discover
    all_pdfs = []
    for url in site_config["listing_urls"]:
        pdfs = discover_pdfs(url, base_url, pdf_pattern)
        all_pdfs.extend(pdfs)
        time.sleep(1)
    
    print(f"  Discovered {len(all_pdfs)} PDFs")
    
    # 2. Download (limited)
    pipeline = IngestionPipeline()
    results = []
    
    for pdf in all_pdfs[:max_pdfs]:
        try:
            local_path = download_pdf(pdf["url"], pdf["title"], name, output_folder)
            # 3. Ingest
            result = pipeline.ingest_file(local_path)
            results.append({"file": pdf["title"][:50], **result})
            print(f"    Ingested: {result['chunks']} chunks, {result['stored']} stored")
        except Exception as e:
            print(f"    Error: {e}")
        time.sleep(2)
    
    return results

# Uncomment to run for a specific site:
# ammc_config = next(s for s in sites if s["name"] == "ammc")
# results = scrape_and_ingest_site(ammc_config, max_pdfs=2)
# pd.DataFrame(results)

## 5. Check Already Collected Documents

In [7]:
# List files already downloaded per source
for folder in DOCS_DIR.iterdir():
    if folder.is_dir():
        pdfs = list(folder.glob("*.pdf"))
        total_size = sum(f.stat().st_size for f in pdfs) / (1024 * 1024)
        print(f"  {folder.name:25s}: {len(pdfs):3d} PDFs ({total_size:.1f} MB)")

  ammc                     : 109 PDFs (376.7 MB)
  bank_al_maghrib          :  41 PDFs (249.4 MB)
  hcp                      :  99 PDFs (880.5 MB)
